# Ü-F — ResolveChoice: die vier Strategien

`loyalty_points` in `customers.json` kommt mal als Zahl (`1200`), mal als Text
(`"gold"`) vor. Ein DynamicFrame kodiert das als **`choice<long,string>`**.
`ResolveChoice` bietet vier Wege, den Konflikt aufzulösen — dieses Notebook
stellt sie **nebeneinander**, damit du die Schemata vergleichen kannst.

Fülle die `# TODO`-Stellen.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.transforms import ResolveChoice
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# Bucket-Name aus deiner Sandbox (tofu output bucket_name) einsetzen:
BUCKET = "gfu-glue-training-629452195361"

## 0) Rohzustand — der Choice-Typ
Lies `customers.json` und sieh dir das abgeleitete Schema an.

In [ ]:
# customers.json direkt per S3-Pfad lesen — KEIN Katalog/Crawler nötig.
# loyalty_points ist mischtypig (1200 vs "gold") -> Glue leitet choice<> ab.
customers = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [f"s3://{BUCKET}/raw/customers/"]},
    format="json",
    transformation_ctx="customers",
)
customers.printSchema()  # loyalty_points: choice<long,string> erwarten

## 1) `cast:long` — auf einen Typ zwingen
Nicht-numerische Werte (`"gold"`) werden zu **NULL**.

In [ ]:
# TODO: ResolveChoice.apply(frame=customers,
#       specs=[("loyalty_points", "cast:long")], transformation_ctx="cast_long")
#       -> cast_long. Danach cast_long.printSchema() und
#       cast_long.toDF().select("customer_id", "loyalty_points").show()

## 2) `make_cols` — je Typ eine eigene Spalte
Aus `loyalty_points` werden `loyalty_points_long` **und** `loyalty_points_string`.

In [ ]:
# TODO: ResolveChoice.apply(frame=customers,
#       specs=[("loyalty_points", "make_cols")], transformation_ctx="make_cols")
#       -> make_cols. Danach make_cols.printSchema() und make_cols.toDF().show()

## 3) `make_struct` — beide Varianten in einem Struct
`loyalty_points` wird zu `struct<long,string>`; pro Zeile ist genau ein Ast gefüllt.

In [ ]:
# TODO: ResolveChoice.apply(frame=customers,
#       specs=[("loyalty_points", "make_struct")], transformation_ctx="make_struct")
#       -> make_struct. Danach printSchema() und toDF().show(truncate=False)

## 4) `project:long` — auf eine Variante projizieren
Behält nur den `long`-Ast; wo `string` stand, bleibt NULL. Wie `cast`, aber ohne Umwandlung.

In [ ]:
# TODO: ResolveChoice.apply(frame=customers,
#       specs=[("loyalty_points", "project:long")], transformation_ctx="project_long")
#       -> project_long. Danach printSchema() und
#       project_long.toDF().select("customer_id", "loyalty_points").show()